[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baluragala/building-rag-pipelines/blob/main/notebooks/04_retrieval.ipynb)

# Building RAG Pipelines
## Notebook 04: Retrieval — Finding the Right Chunks
**Duration:** 30 min &nbsp;|&nbsp; **Mode:** Conceptual + Guided Coding &nbsp;|&nbsp; upGrad Live Session

> Taught **WHY → WHAT → HOW**. We keep asking *"What happens if this step is poorly
> designed?"* and we **predict before we run** and **compare outputs**. LangChain is
> shown as a **parallel mapping** — it abstracts mechanics but not design decisions.

![pipeline](https://dummyimage.com/1000x70/1f2937/ffffff&text=Loading+%E2%86%92+Chunking+%E2%86%92+Retrieval+%E2%86%92+Augmentation+%E2%86%92+Generation+%E2%86%92+Evaluation)

In [ ]:
# ============================================================
# COLAB BOOTSTRAP — run this cell first. (Same as every notebook.)
# ============================================================
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/baluragala/building-rag-pipelines.git"  # INSTRUCTOR: set this

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

_pip("numpy", "openai", "tiktoken", "rank-bm25", "beautifulsoup4", "pypdf")
try:
    import rag_pipeline
except ModuleNotFoundError:
    if IN_COLAB:
        subprocess.run(["git", "clone", "-q", REPO_URL], check=False)
        if os.path.isdir("building-rag-pipelines"):
            sys.path.insert(0, "building-rag-pipelines")
        else:
            print("Clone failed. Upload `rag_pipeline/` + `data/` via the Colab file browser, then re-run.")
    else:
        sys.path.insert(0, os.path.abspath(".."))
    import rag_pipeline

def data_path(*parts):
    for base in ("data", "../data", "building-rag-pipelines/data"):
        p = os.path.join(base, *parts)
        if os.path.exists(p):
            return p
    return os.path.join("data", *parts)

print("rag_pipeline", rag_pipeline.__version__, "ready.  Colab:", IN_COLAB)

In [ ]:
# Providers: OpenAI by default; offline MOCK if no key (class always runs).
import os
if not os.getenv("OPENAI_API_KEY"):
    os.environ["RAG_LLM_PROVIDER"] = "mock"
    os.environ["RAG_EMBED_PROVIDER"] = "mock"
# For the real stack: set OPENAI_API_KEY (getpass or Colab userdata) BEFORE this cell.
from rag_pipeline import config
print(config.current_config())

In [ ]:
from rag_pipeline.loaders import load_directory
docs = load_directory(data_path("corpus"))
print(f"Loaded {len(docs)} documents from the Acme Cloud corpus.")

## WHY — retrieval decides what the model gets to see

Generation cannot fix a retrieval miss. If the relevant chunk never enters the
context, **no prompt on earth recovers it.**

> **What happens if this step is poorly designed?** The classic failure trio:
> (1) relevant chunk **not retrieved** (recall miss), (2) irrelevant chunks
> **crowd it out** (precision miss), (3) the right chunk is present but **ranked
> too low** to survive top-k.

The mental model from the agenda: **optimise recall first, then precision.** Cast
a wide net (hybrid → high recall), then tighten (rerank → high precision). And the
anti-pattern to avoid: **don't rely only on top-k dense similarity.**

## WHAT — embeddings, then four retrieval techniques

- **Embeddings & cosine similarity** — map text to vectors so *distance = dissimilarity*.
- **Vector store** — hold vectors + docs; given a query vector, return the nearest.
- **Dense retrieval** — embed query, cosine top-k. Great at *meaning/paraphrase*.
- **Sparse (BM25)** — classic lexical ranking. Great at *exact terms* (names, codes,
  acronyms like `HTTP 429`) that embeddings blur.
- **Hybrid (BM25 + dense)** — fuse both with **Reciprocal Rank Fusion (RRF)** →
  higher recall than either alone.
- **Metadata filtering** — constrain the search space *before* ranking.
- **Reranking (cross-encoder / LLM)** — re-score a shortlist by reading query+doc
  *together*; far more accurate, so run it only on a small candidate set.

## HOW — embeddings & cosine, with no magic

In [ ]:
from rag_pipeline import config
from rag_pipeline.embeddings import embed_query, cosine_similarity
emb = config.get_embedder()

a = embed_query(emb, "How much is the Growth plan?")
b = embed_query(emb, "What does the Growth subscription cost?")   # paraphrase
c = embed_query(emb, "What encryption does Acme use?")            # unrelated
print(f"cosine(paraphrase)  = {cosine_similarity(a, b):.3f}   <- should be HIGH")
print(f"cosine(unrelated)   = {cosine_similarity(a, c):.3f}   <- should be LOWER")

In [ ]:
# Build the index once for the whole notebook.
from rag_pipeline.chunking import recursive_chunk
from rag_pipeline.vectorstore import InMemoryVectorStore
from rag_pipeline.embeddings import embed_documents
from rag_pipeline.retrieval import DenseRetriever, BM25Retriever, HybridRetriever

chunks = recursive_chunk(docs, chunk_size=500, overlap=50)
store  = InMemoryVectorStore().add(chunks, embed_documents(emb, chunks))

dense  = DenseRetriever(store, emb)
bm25   = BM25Retriever(chunks)
hybrid = HybridRetriever(dense, bm25)
print(f"Indexed {len(chunks)} chunks. Retrievers: dense, BM25, hybrid (RRF).")

> ### ✋ Predict before you run
> We'll ask two questions: (A) 'What HTTP status code is returned when I hit the rate limit?' and (B) 'Can I sign up with a personal Gmail address?'. For which one will BM25 (lexical) beat dense, and for which will dense (semantic) beat BM25? Why?
>
> *Commit to a guess before executing. Comparing prediction vs result is the point.*

In [ ]:
def show(label, retriever, q, k=3):
    hits = retriever.retrieve(q, k=k)
    print(f"  {label:<7}: " + " | ".join(f"{d.metadata['title'][:18]}({s:.2f})" for d, s in hits))

qa = "What HTTP status code is returned when I hit the rate limit?"   # exact token 'HTTP 429'
qb = "Can I sign up with a personal Gmail address?"                    # paraphrase of the docs

print("Q-A (exact term — favours LEXICAL/BM25):", qa)
show("dense", dense, qa); show("bm25", bm25, qa); show("hybrid", hybrid, qa)
print("\nQ-B (paraphrase — favours SEMANTIC/dense):", qb)
show("dense", dense, qb); show("bm25", bm25, qb); show("hybrid", hybrid, qb)

**What you should observe:** BM25 nails Q-A because it matches the literal token
`429`/`HTTP`; dense handles Q-B because 'personal Gmail' paraphrases 'personal
email domains such as gmail.com'. **Hybrid gets both** — which is exactly why
'don't rely only on top-k dense similarity' is on the agenda.

### Metadata filtering — constrain BEFORE ranking

If you already know the answer is in the pricing doc, filtering the search space
first raises precision for free.

In [ ]:
from rag_pipeline.embeddings import embed_query
q = "What is the overage rate for API calls?"
# Only search chunks whose source is the pricing document.
hits = store.search(embed_query(emb, q), k=3,
                    filter_fn=lambda d: "pricing" in d.metadata["source"])
for d, s in hits:
    print(f"{s:.3f}  {d.metadata['title']}  ->  {d.page_content[:80].strip()}")

## HOW — reranking: retrieve wide, rerank narrow

Pull a *wide* candidate set (high recall), then let a reranker read query+doc
together and keep the best few (high precision). We use the LLM reranker here
(no extra model to install); a cross-encoder (`CrossEncoderReranker`) is faster
per query if `sentence-transformers` is available.

In [ ]:
from rag_pipeline.retrieval import LLMReranker, CrossEncoderReranker
llm = config.get_llm()

q = "How often does Acme rotate encryption keys?"
candidates = hybrid.retrieve(q, k=6)          # wide net (recall)
reranked   = LLMReranker(llm).rerank(q, candidates, top_n=3)   # tighten (precision)

print("BEFORE rerank:", [d.metadata['title'][:16] for d, _ in candidates])
print("AFTER  rerank:", [d.metadata['title'][:16] for d, _ in reranked])

## HOW (parallel mapping) — LangChain retrievers

`FAISS.from_documents(...).as_retriever()` wraps embed → store → similarity
search. Same three steps we built by hand. (Requires `langchain-openai` + FAISS.)

In [ ]:
try:
    from rag_pipeline.retrieval import build_langchain_retriever
    lc_ret = build_langchain_retriever(chunks, k=3)
    print("LangChain retriever built:", type(lc_ret).__name__)
except Exception as e:
    print("LangChain/FAISS not installed — concept still holds:", e)

## Recap
- Dense = meaning, BM25 = exact terms, **hybrid = both** (RRF → recall).
- Filter to constrain; **rerank** to sharpen precision on a shortlist.
- **Optimise recall first, then precision.** Never rely on top-k dense alone.

**Next → Notebook 05 (Augmentation & Generation):** how retrieved chunks enter the
prompt, and how to make the LLM answer *grounded* and *cited*.